# 6. Modelo de referencia de árboles de decisión: LightGBM

**Entradas:**
- `dataset_modelo_prep` (`2_Preparación de los datos.ipynb`) y catálogos de *clustering* (`3_Modelado_Categoricas.ipynb`).
- Listas de variables de `config_entradas.json` (`4_Propuesta_Arquitecturas.ipynb`).
- Para la comparación final (sección 10): resultados de `5_Entrenamiento_Modelos.ipynb`.

**Objetivo de este cuaderno:** entrenar un modelo de *gradient boosting* (LightGBM) con las mismas variables tabulares y la misma partición 70-15-15 que las redes neuronales del cuaderno 5, y compararlo con ellas y con la línea base semestral.

**Entorno:** Colab **solo CPU con RAM alta** (LightGBM no usa la GPU en Colab). Puede ejecutarse **en paralelo** con el cuaderno 5, en otra sesión: hasta la sección 9 no depende de él.

| Sección | Contenido |
|---|---|
| 0 | Por qué LightGBM, relación con el cuaderno 5 y reanudación |
| 1 | Librerías, parámetros y rutas en Google Drive |
| 2 | Carga del conjunto, de los catálogos y de las listas de variables |
| 3 | Funciones de carga por partición y matriz de variables |
| 4 | Carga de ENTRENAMIENTO y PRUEBA |
| 5 | Búsqueda de hiperparámetros (evaluada en PRUEBA) |
| 6 | Entrenamiento final e importancia de variables |
| 7 | Evaluación en PRUEBA (15 %) |
| 8 | Validación frente a la línea base (15 %) |
| 9 | Exportación de resultados de LightGBM |
| 10 | **Comparación con las redes neuronales del cuaderno 5** (cuando este haya terminado) |

**Salidas en Google Drive (`Superintendencia/`):** `Modelos_6_LightGBM/`, `Resultados_6_LightGBM/` y `Figuras_6_LightGBM/`. Este cuaderno **no escribe** en las carpetas del cuaderno 5.

## 0. Planteamiento

### 0.1 Por qué LightGBM

En datos tabulares, los ensambles de árboles con *gradient boosting* suelen igualar o superar a las redes neuronales (Grinsztajn, Oyallon y Varoquaux, 2022). Incluir **LightGBM** (Ke et al., 2017) permite responder si la complejidad de las redes se justifica frente a un modelo tabular más simple con datos regulatorios mensuales de baja frecuencia. LightGBM agrupa los valores en histogramas, hace crecer los árboles hoja por hoja y trata de forma nativa los nulos y las variables categóricas.

### 0.2 Mismas condiciones que las redes del cuaderno 5

| Aspecto | Cómo se garantiza |
|---|---|
| Partición 70-15-15 | Mismo hash de `NIU_USUARIO` y mismos umbrales (ENTRENAMIENTO / PRUEBA / VALIDACIÓN). |
| Muestra de entrenamiento y monitoreo | Mismas fracciones (`FRAC_USUARIOS_TRAIN`, `FRAC_MONITOR`, `FRAC_AJUSTE_HP`, `FRAC_PRUEBA_HP`). **Deben coincidir con el cuaderno 5**; la sección 10 lo verifica antes de comparar. |
| Variables | Las 17 numéricas, las 14 de contexto y las 5 categóricas de las arquitecturas 1 y 2 (`config_entradas.json`). |
| Selección | Búsqueda aleatoria puntuada con la MAE en PRUEBA; parada temprana con la MAE del subconjunto de monitoreo. |
| Evaluación | Mismos subconjuntos (todos, comparable con la línea base, consumo cero y positivo) y mismo *bootstrap* por usuario. |

**Diferencias propias de los árboles:** las variables entran **sin escalar** (los árboles solo usan el orden de los valores), los **nulos quedan como `NaN`** (sin banderas) y las **categóricas** van como tipo `category` de pandas (sin *one-hot* ni *embeddings*). No usa la secuencia de 12 meses de la arquitectura 3.

### 0.3 Selección entre familias de modelos

El cuaderno 5 elige la mejor red en PRUEBA y la valida. Este cuaderno valida LightGBM como el mejor de su familia. **La elección entre la mejor red y LightGBM se hace en PRUEBA** (sección 10); en VALIDACIÓN se reporta el desempeño de ambos frente a la línea base y la diferencia entre ellos, sin reajustar nada.

### 0.4 Reanudación tras una desconexión

| Etapa | Qué se guarda en Drive | Al volver a ejecutar |
|---|---|---|
| Vocabulario de categóricas | `vocabularios.json` | Se reutiliza. |
| Búsqueda de hiperparámetros | Una fila por prueba en `01_busqueda_lightgbm.csv` | Se omiten las pruebas completadas. |
| Entrenamiento final | `lightgbm_entrenado.txt` y `lightgbm_resumen.json` | Se omite; si además la búsqueda está completa, **no se carga ENTRENAMIENTO**. Un entrenamiento final interrumpido se repite completo. |
| Predicciones | `cache_predicciones/*.npz` | Se reutilizan si coinciden el modelo y los registros. |

`FORZAR_REINICIO = True` ignora los avances guardados.

## 1. Librerías, parámetros y rutas en Google Drive

In [ ]:
import os
import gc
import json
import time
import glob
import shutil
import hashlib
import warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import dask.dataframe as dd
import matplotlib.pyplot as plt
import lightgbm as lgb
from google.colab import drive

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110
pd.options.display.float_format = '{:,.4f}'.format
PALETA = ['#2C3E50', '#2980B9', '#27AE60', '#E67E22', '#C0392B', '#8E44AD']
print(f'LightGBM {lgb.__version__} | núcleos de CPU: {os.cpu_count()}')

In [ ]:
drive.mount('/content/drive')
DRIVE_RAIZ          = '/content/drive/MyDrive/Superintendencia'
DRIVE_ORIGEN        = os.path.join(DRIVE_RAIZ, 'dataset_modelo_prep')
LOCAL_ORIGEN        = './dataset_modelo_prep'
DIR_CATALOGOS       = os.path.join(DRIVE_RAIZ, 'Catalogos_Categorias')
DIR_MODELOS_PROP    = os.path.join(DRIVE_RAIZ, 'Modelos_4_Propuesta_Arquitecturas')
DIR_MODELOS         = os.path.join(DRIVE_RAIZ, 'Modelos_6_LightGBM')
DIR_RESULTADOS      = os.path.join(DRIVE_RAIZ, 'Resultados_6_LightGBM')
DIR_FIGURAS         = os.path.join(DRIVE_RAIZ, 'Figuras_6_LightGBM')
DIR_CACHE           = os.path.join(DIR_RESULTADOS, 'cache_predicciones')
# Solo lectura: resultados del cuaderno 5 para la comparación final
DIR_MODELOS_NB5     = os.path.join(DRIVE_RAIZ, 'Modelos_5_Entrenamiento')
DIR_RESULTADOS_NB5  = os.path.join(DRIVE_RAIZ, 'Resultados_5_Entrenamiento_Modelos')
for directorio in (DIR_MODELOS, DIR_RESULTADOS, DIR_FIGURAS, DIR_CACHE):
    os.makedirs(directorio, exist_ok=True)

RUTA_MODELO      = os.path.join(DIR_MODELOS, 'lightgbm_entrenado.txt')
RUTA_RESUMEN     = os.path.join(DIR_MODELOS, 'lightgbm_resumen.json')
RUTA_VOCAB       = os.path.join(DIR_MODELOS, 'vocabularios.json')
RUTA_BUSQUEDA    = os.path.join(DIR_RESULTADOS, '01_busqueda_lightgbm.csv')

FORZAR_REINICIO = False   # True = ignorar los avances guardados y empezar de cero
SEMILLA = 42

# 1. Partición 70-15-15 (idéntica a los cuadernos 3 y 5)
HEX_MAX            = 65536
UMBRAL_HEX_TRAIN   = int(round(0.70 * HEX_MAX))   # 45.875
UMBRAL_HEX_PRUEBA  = int(round(0.85 * HEX_MAX))   # 55.706

# 2. Fracciones de usuarios: DEBEN COINCIDIR con el cuaderno 5
FRAC_USUARIOS_TRAIN  = 0.25
FRAC_MONITOR         = 0.10
FRAC_USUARIOS_PRUEBA = 1.00
FRAC_USUARIOS_VALID  = 1.00
FRAC_AJUSTE_HP       = 0.30
FRAC_PRUEBA_HP       = 0.30

# 3. LightGBM
N_PRUEBAS       = 8          # configuraciones de la búsqueda (la primera es LGB_PROPUESTA)
MAX_ARBOLES_HP  = 1000       # árboles máximos por prueba de la búsqueda
MAX_ARBOLES     = 4000       # árboles máximos en el entrenamiento final
PARADA_ARBOLES  = 50         # rondas sin mejora de la MAE de monitoreo antes de detener
LGB_PROPUESTA = {
    'objective': 'huber', 'learning_rate': 0.05, 'num_leaves': 127, 'min_data_in_leaf': 500,
    'feature_fraction': 0.9, 'bagging_fraction': 0.9, 'lambda_l2': 1.0,
}
ESPACIO_LGB = {
    'objective': ['huber', 'regression_l1'],   # huber con alpha = 1 (igual al delta propuesto para las redes) o MAE directa
    'learning_rate': (0.02, 0.2),              # log-uniforme
    'num_leaves': [63, 127, 255],
    'min_data_in_leaf': [100, 500, 2000],
    'feature_fraction': [0.7, 0.9, 1.0],
    'bagging_fraction': [0.7, 0.9, 1.0],
    'lambda_l2': [0.0, 1.0, 10.0],
}
PARAMS_FIJOS = {
    'metric': 'l1', 'alpha': 1.0, 'bagging_freq': 1, 'seed': SEMILLA,
    'deterministic': True, 'force_row_wise': True, 'verbose': -1,
}

# 4. Evaluación (igual que el cuaderno 5)
LIMITE_COMPARABLE_KWH = 100.0
N_BOOTSTRAP           = 1000


def rango_hex(inicio, fin, fraccion):
    return inicio, inicio + int(round((fin - inicio) * fraccion))

RANGOS = {
    'ENTRENAMIENTO': rango_hex(0, UMBRAL_HEX_TRAIN, FRAC_USUARIOS_TRAIN),
    'PRUEBA':        rango_hex(UMBRAL_HEX_TRAIN, UMBRAL_HEX_PRUEBA, FRAC_USUARIOS_PRUEBA),
    'VALIDACION':    rango_hex(UMBRAL_HEX_PRUEBA, HEX_MAX, FRAC_USUARIOS_VALID),
}
HEX_INICIO_MONITOR = RANGOS['ENTRENAMIENTO'][1] - int(round(RANGOS['ENTRENAMIENTO'][1] * FRAC_MONITOR))
pd.DataFrame(RANGOS, index=['hex_desde', 'hex_hasta (excl.)']).T

In [ ]:
def guardar_figura(fig, nombre):
    fig.savefig(os.path.join(DIR_FIGURAS, f'{nombre}.png'), dpi=150, bbox_inches='tight')


def guardar_tabla(df, nombre):
    df.to_csv(os.path.join(DIR_RESULTADOS, f'{nombre}.csv'), encoding='utf-8')


def metricas_regresion(y, y_hat):
    y = np.asarray(y, dtype='float64')
    y_hat = np.asarray(y_hat, dtype='float64')
    ok = np.isfinite(y) & np.isfinite(y_hat)
    y, y_hat = y[ok], y_hat[ok]
    e = y_hat - y
    ss_tot = ((y - y.mean()) ** 2).sum()
    return {
        'n': int(ok.sum()),
        'MAE': np.abs(e).mean(),
        'RMSE': np.sqrt((e ** 2).mean()),
        'R2': 1.0 - (e ** 2).sum() / ss_tot if ss_tot > 0 else np.nan,
        'Sesgo': e.mean(),
    }


def bootstrap_diferencia(usuario, err_a, err_b, n_boot, semilla):
    # Diferencia de MAE por usuario: MAE_b − MAE_a (positivo = A tiene menor error). Cada usuario pesa igual.
    _, inv = np.unique(usuario, return_inverse=True)
    n_reg = np.bincount(inv)
    mae_a = np.bincount(inv, weights=np.abs(err_a)) / n_reg
    mae_b = np.bincount(inv, weights=np.abs(err_b)) / n_reg
    diferencia = mae_b - mae_a
    rng = np.random.default_rng(semilla)
    medias = np.array([diferencia[rng.integers(0, len(diferencia), size=len(diferencia))].mean()
                       for _ in range(n_boot)])
    ic_inf, ic_sup = np.percentile(medias, [2.5, 97.5])
    return {
        'Usuarios': len(diferencia),
        'MAE medio por usuario · A': mae_a.mean(),
        'MAE medio por usuario · B': mae_b.mean(),
        'Diferencia media (B − A)': diferencia.mean(),
        'IC95 inferior': ic_inf,
        'IC95 superior': ic_sup,
        '% usuarios con menor error de A': (diferencia > 0).mean() * 100,
    }

## 2. Carga del conjunto, de los catálogos y de las listas de variables

In [ ]:
# 2.1 Copia local del conjunto particionado por ANIO
archivos_drive = sorted(glob.glob(os.path.join(DRIVE_ORIGEN, '**', '*.parquet'), recursive=True))
print(f'Archivos parquet encontrados en Drive: {len(archivos_drive):,}')

def _copiar_preservando_particion(ruta_origen):
    rel = os.path.relpath(ruta_origen, DRIVE_ORIGEN)
    ruta_destino = os.path.join(LOCAL_ORIGEN, rel)
    os.makedirs(os.path.dirname(ruta_destino), exist_ok=True)
    if not os.path.exists(ruta_destino):
        shutil.copy2(ruta_origen, ruta_destino)

with ThreadPoolExecutor(max_workers=8) as ex:
    list(ex.map(_copiar_preservando_particion, archivos_drive))
print('Copia local completada en:', LOCAL_ORIGEN)

In [ ]:
# 2.2 Variables de las arquitecturas 1 y 2 (cuaderno 4)
with open(os.path.join(DIR_MODELOS_PROP, 'config_entradas.json'), encoding='utf-8') as fh:
    config = json.load(fh)
NUM_TABULAR = config['num_tabular']
CTX_TRAFO   = config['ctx_trafo']
CTX_CIRC    = config['ctx_circ']
CTX         = CTX_TRAFO + CTX_CIRC
CATEGORICAS = config['categoricas']
COLS_MODELO = NUM_TABULAR + CTX + list(CATEGORICAS)
print(f'{len(NUM_TABULAR)} numéricas + {len(CTX)} de contexto + {len(CATEGORICAS)} categóricas = {len(COLS_MODELO)} variables')


# 2.3 Catálogos del cuaderno 3 (misma construcción que el cuaderno 5)
def limpiar_circuito(serie):
    return serie.astype(str).str.strip().replace({'': 'SIN_CIRCUITO'})

cat_trafo = pd.read_parquet(os.path.join(DIR_CATALOGOS, 'catalogo_clusters_transformador.parquet'))
cat_circ  = pd.read_parquet(os.path.join(DIR_CATALOGOS, 'catalogo_clusters_circuito.parquet'))

tabla_trafo = pd.DataFrame(index=pd.Index(cat_trafo['COD_TRANSFORMADOR'].astype(str), name='COD_TRANSFORMADOR'))
for col in CTX_TRAFO:
    origen = col.replace('TRAFO_', '', 1)
    if origen == 'LOG_N_USUARIOS':
        tabla_trafo[col] = np.log1p(cat_trafo['N_USUARIOS'].to_numpy(dtype='float64'))
    else:
        tabla_trafo[col] = cat_trafo[origen].to_numpy(dtype='float64')
tabla_trafo['CLUSTER_TRANSFORMADOR'] = cat_trafo['CLUSTER_TRANSFORMADOR'].astype(str).to_numpy()
tabla_trafo['CLUSTER_CIRCUITO'] = cat_trafo['CLUSTER_CIRCUITO'].astype(str).to_numpy()
tabla_trafo = tabla_trafo.loc[~tabla_trafo.index.duplicated(keep='first')]

tabla_circ = pd.DataFrame(index=pd.Index(limpiar_circuito(cat_circ['COD_CIRCUITO_LINEA']), name='COD_CIRCUITO_CLEAN'))
for col in CTX_CIRC:
    tabla_circ[col] = cat_circ[col.replace('CIRC_', '', 1)].to_numpy(dtype='float64')
tabla_circ = tabla_circ.loc[~tabla_circ.index.duplicated(keep='first')]
print(f'Transformadores en catálogo: {len(tabla_trafo):,} | circuitos en catálogo: {len(tabla_circ):,}')

## 3. Funciones de carga por partición y matriz de variables

- `cargar_particion` y `enriquecer` son las del cuaderno 5, sin la secuencia de 12 meses.
- `construir_matriz` devuelve un `DataFrame` con las 36 variables: numéricas y de contexto en `float32` (nulos como `NaN`) y categóricas como `category` con un **vocabulario fijo**, de modo que los códigos son idénticos en las tres particiones. Los clústeres usan el vocabulario del catálogo y `TIPO_MEDIDOR` el de ENTRENAMIENTO (se guarda en `vocabularios.json`).

In [ ]:
COLS_LECTURA = [
    'PERIODO', 'NIU_USUARIO', 'KWH_DIA', 'BENCH_KWH_DIA', 'BENCH_VALIDO',
    'ESTRATO_SECTOR', 'TIPO_MEDIDOR', 'MES_CONSUMO', 'COD_TRANSFORMADOR', 'COD_CIRCUITO_LINEA',
] + NUM_TABULAR


def cargar_particion(nombre):
    h_desde, h_hasta = RANGOS[nombre]
    ddf = dd.read_parquet(LOCAL_ORIGEN, columns=COLS_LECTURA)
    prefijo = ddf['NIU_USUARIO'].str.slice(0, 4).str.lower()
    filtro = prefijo >= f'{h_desde:04x}'
    if h_hasta < HEX_MAX:
        filtro = filtro & (prefijo < f'{h_hasta:04x}')
    pdf = ddf[filtro].compute().reset_index(drop=True)
    pdf = pdf.loc[pdf['KWH_DIA'].notna()].reset_index(drop=True)

    pdf['HEX'] = pdf['NIU_USUARIO'].str.slice(0, 4).map(lambda s: int(s, 16)).astype('int32')
    assert pdf['HEX'].between(h_desde, h_hasta - 1).all(), 'Registros fuera del rango de la partición'
    pdf['ANIO'] = pd.to_datetime(pdf['PERIODO']).dt.year.astype('int16')
    pdf['USUARIO'] = pd.factorize(pdf['NIU_USUARIO'])[0].astype('int32')
    print(f'{nombre}: {len(pdf):,} registros | {pdf["USUARIO"].max() + 1:,} usuarios | hex [{h_desde}, {h_hasta})')
    return pdf


def enriquecer(pdf):
    ctx_t = tabla_trafo.reindex(pdf['COD_TRANSFORMADOR'].astype(str))
    for col in CTX_TRAFO:
        pdf[col] = ctx_t[col].to_numpy()
    for col in ['CLUSTER_TRANSFORMADOR', 'CLUSTER_CIRCUITO']:
        pdf[col] = pd.Categorical(ctx_t[col].to_numpy())
    ctx_c = tabla_circ.reindex(limpiar_circuito(pdf['COD_CIRCUITO_LINEA']))
    for col in CTX_CIRC:
        pdf[col] = ctx_c[col].to_numpy()
    print(f"% con transformador en catálogo: {pdf['CLUSTER_TRANSFORMADOR'].notna().mean() * 100:.2f} | "
          f"% con circuito en catálogo: {pdf[CTX_CIRC[0]].notna().mean() * 100:.2f}")
    return pdf


def construir_matriz(pdf, vocab):
    X = pd.DataFrame(index=pdf.index)
    for col in NUM_TABULAR + CTX:
        X[col] = pdf[col].to_numpy(dtype='float32', na_value=np.nan)
    for col in CATEGORICAS:
        if col in ('ESTRATO_SECTOR', 'MES_CONSUMO'):
            valores = pdf[col].to_numpy(dtype='float64', na_value=np.nan)
            X[col] = pd.Categorical(valores, categories=[float(v) for v in vocab[col]])
        else:
            X[col] = pd.Categorical(pdf[col].astype('object'), categories=vocab[col])
    datos = {
        'X': X,
        'y': pdf['KWH_DIA'].to_numpy(dtype='float32'),
        'bench': pdf['BENCH_KWH_DIA'].to_numpy(dtype='float32', na_value=np.nan),
        'bench_valido': pdf['BENCH_VALIDO'].to_numpy(dtype='float32', na_value=0).astype('int8'),
        'estrato': pdf['ESTRATO_SECTOR'].to_numpy(dtype='float32', na_value=0).astype('int8'),
        'anio': pdf['ANIO'].to_numpy(),
        'usuario': pdf['USUARIO'].to_numpy(),
        'hex': pdf['HEX'].to_numpy(),
        'meta': pdf[['NIU_USUARIO', 'PERIODO']].reset_index(drop=True),
    }
    return datos


def mascara_comparable(D):
    return ((D['bench_valido'] == 1) & np.isfinite(D['bench'])
            & (D['y'] <= LIMITE_COMPARABLE_KWH) & (D['bench'] <= LIMITE_COMPARABLE_KWH))


def predecir_con_cache(D, particion):
    # Reutiliza las predicciones guardadas si coinciden los registros y el archivo del modelo
    firma = (hashlib.sha1(D['y'].tobytes()).hexdigest()
             + f'|{os.path.getsize(RUTA_MODELO)}|{os.path.getmtime(RUTA_MODELO):.0f}')
    ruta_cache = os.path.join(DIR_CACHE, f'pred_{particion}_lightgbm.npz')
    if os.path.exists(ruta_cache) and not FORZAR_REINICIO:
        guardado = np.load(ruta_cache, allow_pickle=False)
        if str(guardado['firma']) == firma:
            print(f'{particion}: predicciones leídas de la caché en Drive.')
            return guardado['kwh_dia']
    inicio = time.time()
    booster = lgb.Booster(model_file=RUTA_MODELO)
    n, bloque = len(D['y']), 2_000_000
    y_hat = np.empty(n, dtype='float32')
    for i in range(0, n, bloque):   # por bloques para no convertir toda la partición a float64 de una vez
        y_hat[i:i + bloque] = booster.predict(D['X'].iloc[i:i + bloque])
    np.savez(ruta_cache, firma=np.array(firma), kwh_dia=y_hat)
    print(f'{particion}: predicción en {(time.time() - inicio) / 60:.1f} min')
    return y_hat

## 4. Carga de ENTRENAMIENTO y PRUEBA

Si el modelo final y la búsqueda ya están completos en Drive, ENTRENAMIENTO no se carga.

In [ ]:
def busqueda_completa():
    if FORZAR_REINICIO or not os.path.exists(RUTA_BUSQUEDA):
        return False
    return set(range(N_PRUEBAS)).issubset(set(pd.read_csv(RUTA_BUSQUEDA)['Prueba'].astype(int)))

MODELO_COMPLETO = (not FORZAR_REINICIO and os.path.exists(RUTA_MODELO) and os.path.exists(RUTA_RESUMEN)
                   and os.path.exists(RUTA_VOCAB) and busqueda_completa())

if MODELO_COMPLETO:
    print('LightGBM ya está entrenado en Drive: no se carga ENTRENAMIENTO.')
    with open(RUTA_VOCAB, encoding='utf-8') as fh:
        VOCAB = json.load(fh)
    D_train = None
else:
    pdf_train = enriquecer(cargar_particion('ENTRENAMIENTO'))
    if os.path.exists(RUTA_VOCAB) and not FORZAR_REINICIO:
        with open(RUTA_VOCAB, encoding='utf-8') as fh:
            VOCAB = json.load(fh)
    else:
        VOCAB = {
            'ESTRATO_SECTOR': list(range(1, 7)),
            'TIPO_MEDIDOR': sorted(pdf_train['TIPO_MEDIDOR'].dropna().astype(str).unique()),
            'MES_CONSUMO': list(range(1, 13)),
            'CLUSTER_TRANSFORMADOR': sorted(tabla_trafo['CLUSTER_TRANSFORMADOR'].unique()),
            'CLUSTER_CIRCUITO': sorted(tabla_trafo['CLUSTER_CIRCUITO'].unique()),
        }
        with open(RUTA_VOCAB, 'w', encoding='utf-8') as fh:
            json.dump(VOCAB, fh, ensure_ascii=False, indent=2)
    D_train = construir_matriz(pdf_train, VOCAB)
    del pdf_train
    gc.collect()

    es_monitor = D_train['hex'] >= HEX_INICIO_MONITOR
    idx_ajuste = np.nonzero(~es_monitor)[0]
    idx_monitor = np.nonzero(es_monitor)[0]
    idx_ajuste_hp = idx_ajuste[D_train['hex'][idx_ajuste] < int(round(HEX_INICIO_MONITOR * FRAC_AJUSTE_HP))]
    display(pd.Series({
        'Registros de ajuste': len(idx_ajuste),
        'Registros de monitoreo (parada temprana)': len(idx_monitor),
        'Registros de ajuste por prueba de la búsqueda': len(idx_ajuste_hp),
        'Memoria de la matriz (GB)': D_train['X'].memory_usage(deep=True).sum() / 1024 ** 3,
    }, name='ENTRENAMIENTO').to_frame())

In [ ]:
D_prueba = construir_matriz(enriquecer(cargar_particion('PRUEBA')), VOCAB)
h0, h1 = RANGOS['PRUEBA']
idx_prueba_hp = np.nonzero(D_prueba['hex'] < h0 + int(round((h1 - h0) * FRAC_PRUEBA_HP)))[0]
print(f'Registros de PRUEBA para puntuar la búsqueda: {len(idx_prueba_hp):,}')
_ = gc.collect()

## 5. Búsqueda de hiperparámetros (evaluada en PRUEBA)

Búsqueda aleatoria (Bergstra y Bengio, 2012) con el mismo protocolo de las redes del cuaderno 5: `N_PRUEBAS` configuraciones (la primera es la propuesta), cada una entrenada con `FRAC_AJUSTE_HP` de los usuarios de ajuste, con parada temprana sobre la **MAE del subconjunto de monitoreo** (`PARADA_ARBOLES` rondas sin mejora, hasta `MAX_ARBOLES_HP` árboles), y puntuada con la **MAE en PRUEBA**.

| Hiperparámetro | Espacio | Propuesta |
|---|---|---|
| Función objetivo | Huber (α = 1) o MAE (`regression_l1`) | Huber |
| Tasa de aprendizaje | log-uniforme [0,02; 0,2] | 0,05 |
| Hojas por árbol (`num_leaves`) | {63, 127, 255} | 127 |
| Mínimo de registros por hoja | {100, 500, 2.000} | 500 |
| Fracción de variables por árbol | {0,7; 0,9; 1,0} | 0,9 |
| Fracción de registros por árbol (*bagging*) | {0,7; 0,9; 1,0} | 0,9 |
| Regularización L2 | {0; 1; 10} | 1 |

In [ ]:
def muestrear_hiperparametros(n_pruebas, semilla):
    rng = np.random.default_rng(semilla)
    pruebas = [dict(LGB_PROPUESTA)]
    lo, hi = ESPACIO_LGB['learning_rate']
    while len(pruebas) < n_pruebas:
        hp = {}
        for clave, espacio in ESPACIO_LGB.items():
            if clave == 'learning_rate':
                hp[clave] = float(10 ** rng.uniform(np.log10(lo), np.log10(hi)))
            else:
                hp[clave] = espacio[int(rng.integers(len(espacio)))]
        pruebas.append(hp)
    return pruebas


def entrenar(hp, idx_fit, max_arboles):
    ds_ajuste = lgb.Dataset(D_train['X'].iloc[idx_fit], label=D_train['y'][idx_fit], free_raw_data=True)
    ds_monitor = lgb.Dataset(D_train['X'].iloc[idx_monitor], label=D_train['y'][idx_monitor], reference=ds_ajuste)
    return lgb.train({**PARAMS_FIJOS, **hp}, ds_ajuste, num_boost_round=max_arboles,
                     valid_sets=[ds_monitor], valid_names=['monitoreo'],
                     callbacks=[lgb.early_stopping(PARADA_ARBOLES, verbose=False)])


PRUEBAS = muestrear_hiperparametros(N_PRUEBAS, SEMILLA + 100)
pd.DataFrame(PRUEBAS).rename_axis('Prueba')

In [ ]:
resultados = pd.read_csv(RUTA_BUSQUEDA) if (os.path.exists(RUTA_BUSQUEDA) and not FORZAR_REINICIO) else pd.DataFrame()

for _, fila in resultados.iterrows():
    k = int(fila['Prueba'])
    if k < len(PRUEBAS):
        for clave, valor in PRUEBAS[k].items():
            igual = fila[clave] == valor if isinstance(valor, str) else np.isclose(float(fila[clave]), float(valor))
            if not igual:
                raise ValueError(f'La prueba {k} guardada en Drive no coincide con ESPACIO_LGB/LGB_PROPUESTA actuales. '
                                 'Restaure los parámetros o use FORZAR_REINICIO = True.')

hechas = set(resultados.get('Prueba', pd.Series(dtype=int)).astype(int))
pendientes = [k for k in range(len(PRUEBAS)) if k not in hechas]
print(f'Pruebas completadas en sesiones anteriores: {len(hechas)} | pendientes: {len(pendientes)}')

idx_hp_ord = np.sort(idx_prueba_hp)
y_prueba_hp = D_prueba['y'][idx_hp_ord]
for k in pendientes:
    hp = PRUEBAS[k]
    inicio = time.time()
    booster = entrenar(hp, idx_ajuste_hp, MAX_ARBOLES_HP)
    y_hat = booster.predict(D_prueba['X'].iloc[idx_hp_ord], num_iteration=booster.best_iteration)
    fila = {'Prueba': k, 'Es_propuesta': k == 0, **hp, 'Arboles': int(booster.best_iteration),
            **{f'{m}_PRUEBA': v for m, v in metricas_regresion(y_prueba_hp, y_hat).items() if m != 'n'},
            'Minutos': (time.time() - inicio) / 60}
    resultados = pd.concat([resultados, pd.DataFrame([fila])], ignore_index=True)
    resultados.to_csv(RUTA_BUSQUEDA, index=False, encoding='utf-8')
    print(f"prueba {k} | MAE {fila['MAE_PRUEBA']:.4f} | {fila['Arboles']} árboles | {fila['Minutos']:.1f} min")
    del booster
    gc.collect()

resultados.sort_values('MAE_PRUEBA').set_index('Prueba')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(resultados['learning_rate'], resultados['MAE_PRUEBA'], color=PALETA[1], s=50)
prop = resultados.loc[resultados['Prueba'] == 0]
ax.scatter(prop['learning_rate'], prop['MAE_PRUEBA'], facecolors='none', edgecolors='black', s=140, linewidths=1.2,
           label='Propuesta')
ax.set_xscale('log')
ax.set_title('LightGBM: tasa de aprendizaje frente a MAE en PRUEBA')
ax.set_xlabel('Tasa de aprendizaje (escala log)')
ax.set_ylabel('MAE (kWh/día)')
ax.legend()
plt.tight_layout()
guardar_figura(fig, '01_busqueda_hiperparametros')
plt.show()

## 6. Entrenamiento final e importancia de variables

Se entrena con **toda** la muestra de ajuste y los mejores hiperparámetros de la búsqueda, hasta `MAX_ARBOLES` árboles con parada temprana, y se guarda el modelo con los árboles de la mejor iteración.

In [ ]:
mejor_fila = resultados.loc[resultados['MAE_PRUEBA'].idxmin()]
MEJORES_HP = {}
for clave, valor in LGB_PROPUESTA.items():
    if isinstance(valor, str):
        MEJORES_HP[clave] = str(mejor_fila[clave])
    elif isinstance(valor, int):
        MEJORES_HP[clave] = int(mejor_fila[clave])
    else:
        MEJORES_HP[clave] = float(mejor_fila[clave])
print('Mejores hiperparámetros:', MEJORES_HP)

if MODELO_COMPLETO or (os.path.exists(RUTA_MODELO) and os.path.exists(RUTA_RESUMEN) and not FORZAR_REINICIO):
    with open(RUTA_RESUMEN, encoding='utf-8') as fh:
        resumen_modelo = json.load(fh)
    if resumen_modelo['hiperparametros'] != MEJORES_HP:
        raise ValueError('El modelo guardado se entrenó con otros hiperparámetros. Use FORZAR_REINICIO = True.')
    print('Modelo final entrenado en una sesión anterior: se omite.')
else:
    inicio = time.time()
    booster = entrenar(MEJORES_HP, idx_ajuste, MAX_ARBOLES)
    booster.save_model(RUTA_MODELO, num_iteration=booster.best_iteration)
    resumen_modelo = {
        'hiperparametros': MEJORES_HP,
        'arboles': int(booster.best_iteration),
        'mae_monitoreo': float(booster.best_score['monitoreo']['l1']),
        'minutos': (time.time() - inicio) / 60,
    }
    with open(RUTA_RESUMEN, 'w', encoding='utf-8') as fh:
        json.dump(resumen_modelo, fh, ensure_ascii=False, indent=2)
    del booster
    gc.collect()

tabla_modelo = pd.Series({
    **MEJORES_HP,
    'MAE propuesta (búsqueda)': resultados.loc[resultados['Prueba'] == 0, 'MAE_PRUEBA'].iloc[0],
    'MAE mejor (búsqueda)': mejor_fila['MAE_PRUEBA'],
    'Árboles (mejor iteración)': resumen_modelo['arboles'],
    'MAE de monitoreo': resumen_modelo['mae_monitoreo'],
    'Minutos': resumen_modelo['minutos'],
}, name='lightgbm').to_frame()
guardar_tabla(tabla_modelo, '02_resumen_modelo')

# Liberar la memoria de ENTRENAMIENTO
D_train = None
_ = gc.collect()
tabla_modelo

In [ ]:
# Importancia de variables: ganancia total de las divisiones en que interviene cada variable
booster = lgb.Booster(model_file=RUTA_MODELO)
importancia = pd.DataFrame({
    'Variable': booster.feature_name(),
    'Ganancia': booster.feature_importance(importance_type='gain'),
    'Divisiones': booster.feature_importance(importance_type='split'),
})
importancia['% Ganancia'] = importancia['Ganancia'] / importancia['Ganancia'].sum() * 100
importancia = importancia.sort_values('% Ganancia', ascending=False).set_index('Variable')
guardar_tabla(importancia, '03_importancia_variables')

top = importancia.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6.5))
ax.barh(top.index, top['% Ganancia'], color=PALETA[1])
ax.set_title('LightGBM: importancia de variables (% de la ganancia total, 20 primeras)')
ax.set_xlabel('% de la ganancia')
plt.tight_layout()
guardar_figura(fig, '02_importancia_variables')
plt.show()
del booster
importancia.head(20)

## 7. Evaluación en PRUEBA (15 %)

Mismos subconjuntos que el cuaderno 5: todos los registros, comparable con la línea base (`BENCH_VALIDO == 1`, real y línea base ≤ 100 kWh/día), consumo real cero y consumo real positivo.

In [ ]:
def tabla_metricas(D, predicciones):
    comparable = mascara_comparable(D)
    es_cero = D['y'] == 0
    filas = []
    for subconjunto, mascara in [('Todos los registros', np.ones_like(comparable)),
                                 ('Comparable con la línea base', comparable),
                                 ('Consumo real cero', es_cero),
                                 ('Consumo real > 0', ~es_cero)]:
        for nombre, y_hat in predicciones.items():
            filas.append({'Subconjunto': subconjunto, 'Modelo': nombre,
                          **metricas_regresion(D['y'][mascara], y_hat[mascara])})
    return pd.DataFrame(filas).set_index(['Subconjunto', 'Modelo'])


y_hat_prueba = predecir_con_cache(D_prueba, 'PRUEBA')
tabla_prueba = tabla_metricas(D_prueba, {'lightgbm': y_hat_prueba, 'LINEA_BASE': D_prueba['bench']})
guardar_tabla(tabla_prueba, '04_metricas_prueba')
tabla_prueba

In [ ]:
D_prueba = None
_ = gc.collect()

## 8. Validación frente a la línea base (15 %)

In [ ]:
D_valid = construir_matriz(enriquecer(cargar_particion('VALIDACION')), VOCAB)
y_hat_valid = predecir_con_cache(D_valid, 'VALIDACION')
comparable_valid = mascara_comparable(D_valid)

tabla_valid = tabla_metricas(D_valid, {'lightgbm': y_hat_valid, 'LINEA_BASE': D_valid['bench']})
guardar_tabla(tabla_valid, '05_validacion_vs_linea_base')
display(tabla_valid)

m = comparable_valid
bootstrap_base = bootstrap_diferencia(D_valid['usuario'][m], y_hat_valid[m] - D_valid['y'][m],
                                      D_valid['bench'][m] - D_valid['y'][m], N_BOOTSTRAP, SEMILLA)
tabla_bootstrap = pd.Series(bootstrap_base, name='A = lightgbm, B = línea base').to_frame()
guardar_tabla(tabla_bootstrap, '06_bootstrap_vs_linea_base')
tabla_bootstrap

In [ ]:
def metricas_por_grupo(grupo, etiqueta, predicciones):
    filas = []
    for valor in np.unique(grupo[comparable_valid]):
        mm = comparable_valid & (grupo == valor)
        for nombre, y_hat in predicciones.items():
            r = metricas_regresion(D_valid['y'][mm], y_hat[mm])
            filas.append({etiqueta: valor, 'Modelo': nombre, 'MAE': r['MAE'], 'RMSE': r['RMSE'], 'R2': r['R2']})
    return pd.DataFrame(filas).pivot(index=etiqueta, columns='Modelo', values=['MAE', 'RMSE', 'R2'])

pred_valid = {'lightgbm': y_hat_valid, 'LINEA_BASE': D_valid['bench']}
valid_por_anio = metricas_por_grupo(D_valid['anio'], 'Año', pred_valid)
valid_por_estrato = metricas_por_grupo(D_valid['estrato'], 'Estrato', pred_valid)
guardar_tabla(valid_por_anio, '07_validacion_por_anio')
guardar_tabla(valid_por_estrato, '08_validacion_por_estrato')
display(valid_por_anio)
display(valid_por_estrato)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
lim = float(np.percentile(D_valid['y'][comparable_valid], 99))
for ax, (nombre, y_hat) in zip(axes, pred_valid.items()):
    mm = comparable_valid & np.isfinite(y_hat)
    hb = ax.hexbin(D_valid['y'][mm], y_hat[mm], gridsize=60, extent=(0, lim, 0, lim), bins='log', cmap='Blues', mincnt=1)
    ax.plot([0, lim], [0, lim], color=PALETA[4], linestyle='--', linewidth=1)
    ax.set_title(f'{nombre} (VALIDACIÓN, comparable)')
    ax.set_xlabel('KWH_DIA real')
    ax.set_ylabel('KWH_DIA estimado')
    fig.colorbar(hb, ax=ax, label='Registros (escala log)')
plt.tight_layout()
guardar_figura(fig, '03_validacion_vs_linea_base')
plt.show()

## 9. Exportación de resultados de LightGBM

In [ ]:
predicciones_valid = D_valid['meta'].assign(
    KWH_DIA=D_valid['y'], KWH_DIA_ESTIMADO=y_hat_valid, BENCH_KWH_DIA=D_valid['bench'],
    COMPARABLE=comparable_valid.astype('int8'),
)
predicciones_valid.to_parquet(os.path.join(DIR_RESULTADOS, 'predicciones_validacion_lightgbm.parquet'), index=False)

resumen = {
    'modelo': 'lightgbm',
    'hiperparametros': MEJORES_HP,
    'entrenamiento': resumen_modelo,
    'particion': {'rangos_hex': {k: list(v) for k, v in RANGOS.items()}, 'hex_inicio_monitor': HEX_INICIO_MONITOR},
    'validacion_comparable': tabla_valid.loc['Comparable con la línea base'].to_dict(orient='index'),
    'bootstrap_vs_linea_base': bootstrap_base,
}
with open(os.path.join(DIR_RESULTADOS, 'resumen_lightgbm.json'), 'w', encoding='utf-8') as fh:
    json.dump(resumen, fh, ensure_ascii=False, indent=2, default=float)
print('Resultados de LightGBM guardados en', DIR_RESULTADOS)

## 10. Comparación con las redes neuronales del cuaderno 5

Esta sección necesita que `5_Entrenamiento_Modelos.ipynb` haya terminado. Si aún no termina, se muestra un aviso; basta con volver a ejecutar el cuaderno cuando termine: el modelo y las predicciones de LightGBM se leen de Drive y no se recalculan.

1. **Verificación:** la partición y las fracciones de usuarios del cuaderno 5 (`preprocesamiento.json`) deben ser idénticas a las de este cuaderno, y los registros de PRUEBA deben coincidir (mismo número de registros de la línea base).
2. **Elección entre familias en PRUEBA:** tabla conjunta de las tres redes, LightGBM y la línea base; gana el menor MAE sobre todos los registros.
3. **VALIDACIÓN:** se emparejan registro a registro las predicciones de LightGBM y de la red seleccionada en el cuaderno 5 (`NIU_USUARIO`, `PERIODO`), se comparan con la línea base y entre sí, con *bootstrap* por usuario.

In [ ]:
ruta_prep_nb5 = os.path.join(DIR_MODELOS_NB5, 'preprocesamiento.json')
ruta_tabla_nb5 = os.path.join(DIR_RESULTADOS_NB5, '03_comparacion_modelos_prueba.csv')
ruta_resumen_nb5 = os.path.join(DIR_RESULTADOS_NB5, 'resumen_entrenamiento.json')
NB5_DISPONIBLE = all(os.path.exists(r) for r in (ruta_prep_nb5, ruta_tabla_nb5, ruta_resumen_nb5))

if not NB5_DISPONIBLE:
    print('El cuaderno 5 todavía no ha terminado: vuelva a ejecutar este cuaderno cuando termine '
          '(LightGBM no se reentrena; sus resultados se leen de Drive).')
else:
    with open(ruta_prep_nb5, encoding='utf-8') as fh:
        particion_nb5 = json.load(fh)['particion']
    if particion_nb5 != resumen['particion']:
        raise ValueError('La partición o las fracciones de usuarios del cuaderno 5 no coinciden con las de este cuaderno; '
                         'la comparación no sería válida. Iguale FRAC_USUARIOS_TRAIN, FRAC_MONITOR y las demás fracciones.')
    with open(ruta_resumen_nb5, encoding='utf-8') as fh:
        resumen_nb5 = json.load(fh)

    tabla_nb5 = pd.read_csv(ruta_tabla_nb5, index_col=[0, 1])
    n_base_nb5 = tabla_nb5.loc[('Todos los registros', 'LINEA_BASE'), 'n']
    n_base_aqui = tabla_prueba.loc[('Todos los registros', 'LINEA_BASE'), 'n']
    assert n_base_nb5 == n_base_aqui, f'PRUEBA no coincide entre cuadernos ({n_base_nb5} vs {n_base_aqui} registros)'

    tabla_conjunta = pd.concat([tabla_nb5, tabla_prueba.drop(index='LINEA_BASE', level='Modelo')]).sort_index(level=0, sort_remaining=False)
    guardar_tabla(tabla_conjunta, '09_comparacion_prueba_redes_y_lightgbm')

    candidatos = tabla_conjunta.loc['Todos los registros'].drop(index='LINEA_BASE')
    GANADOR_PRUEBA = candidatos['MAE'].idxmin()
    print(f"Mejor red (cuaderno 5): {resumen_nb5['modelo_seleccionado']}")
    print(f'Menor MAE en PRUEBA entre todas las familias: {GANADOR_PRUEBA}')
    display(candidatos.sort_values('MAE'))

In [ ]:
if NB5_DISPONIBLE:
    red = resumen_nb5['modelo_seleccionado']
    ruta_pred_red = os.path.join(DIR_RESULTADOS_NB5, f'predicciones_validacion_{red}.parquet')
    if not os.path.exists(ruta_pred_red):
        raise FileNotFoundError(f'No está {ruta_pred_red}. Ejecute el cuaderno 5 con GUARDAR_PREDICCIONES_VALID = True.')

    # Emparejar registro a registro (con un contador para filas repetidas del mismo usuario y periodo)
    def con_clave(df):
        df = df.copy()
        df['PERIODO'] = pd.to_datetime(df['PERIODO'])
        df['DUP'] = df.groupby(['NIU_USUARIO', 'PERIODO']).cumcount()
        return df

    pred_red = con_clave(pd.read_parquet(ruta_pred_red))
    pred_lgb = con_clave(predicciones_valid)
    pares = pred_lgb.merge(pred_red[['NIU_USUARIO', 'PERIODO', 'DUP', 'KWH_DIA', 'KWH_DIA_ESTIMADO']],
                           on=['NIU_USUARIO', 'PERIODO', 'DUP'], suffixes=('', '_RED'), how='inner')
    assert len(pares) == len(pred_lgb) == len(pred_red), 'VALIDACIÓN no coincide entre cuadernos'
    assert np.allclose(pares['KWH_DIA'], pares['KWH_DIA_RED'], atol=1e-4), 'El objetivo no coincide entre cuadernos'

    y = pares['KWH_DIA'].to_numpy(dtype='float64')
    comparable = pares['COMPARABLE'].to_numpy() == 1
    predicciones_finales = {
        'lightgbm': pares['KWH_DIA_ESTIMADO'].to_numpy(),
        red: pares['KWH_DIA_ESTIMADO_RED'].to_numpy(),
        'LINEA_BASE': pares['BENCH_KWH_DIA'].to_numpy(),
    }
    filas = []
    for subconjunto, mascara in [('Todos los registros', np.ones_like(comparable)),
                                 ('Comparable con la línea base', comparable)]:
        for nombre, y_hat in predicciones_finales.items():
            filas.append({'Subconjunto': subconjunto, 'Modelo': nombre, **metricas_regresion(y[mascara], y_hat[mascara])})
    tabla_final = pd.DataFrame(filas).set_index(['Subconjunto', 'Modelo'])
    guardar_tabla(tabla_final, '10_validacion_redes_y_lightgbm')
    display(tabla_final)

    usuarios = pd.factorize(pares['NIU_USUARIO'])[0]
    b = bootstrap_diferencia(usuarios[comparable],
                             predicciones_finales['lightgbm'][comparable] - y[comparable],
                             predicciones_finales[red][comparable] - y[comparable], N_BOOTSTRAP, SEMILLA)
    tabla_lgb_vs_red = pd.Series(b, name=f'A = lightgbm, B = {red}').to_frame()
    guardar_tabla(tabla_lgb_vs_red, '11_bootstrap_lightgbm_vs_red')
    display(tabla_lgb_vs_red)

    if b['IC95 inferior'] > 0:
        veredicto = f'LightGBM tiene menor error que {red} en VALIDACIÓN (IC95 por encima de 0).'
    elif b['IC95 superior'] < 0:
        veredicto = f'{red} tiene menor error que LightGBM en VALIDACIÓN (IC95 por debajo de 0).'
    else:
        veredicto = f'No hay diferencia concluyente entre LightGBM y {red} en VALIDACIÓN (el IC95 incluye 0).'
    print(f'Elegido en PRUEBA (criterio del protocolo): {GANADOR_PRUEBA}')
    print('Comparación en VALIDACIÓN:', veredicto)

    with open(os.path.join(DIR_RESULTADOS, 'resumen_comparacion_familias.json'), 'w', encoding='utf-8') as fh:
        json.dump({'ganador_prueba': GANADOR_PRUEBA, 'red_seleccionada_nb5': red,
                   'bootstrap_lightgbm_vs_red': b, 'veredicto_validacion': veredicto},
                  fh, ensure_ascii=False, indent=2, default=float)

### Conclusiones

*(Completar tras la ejecución con las tablas de `Resultados_6_LightGBM`.)*

1. **Búsqueda de hiperparámetros:** mejora frente a la configuración propuesta (tabla `02`).
2. **Variables más importantes** para los árboles (tabla `03`) y su relación con lo encontrado en los cuadernos 2 y 3.
3. **LightGBM frente a la línea base** en VALIDACIÓN (tablas `05` a `08`).
4. **Redes neuronales frente a LightGBM:** modelo elegido en PRUEBA (tabla `09`) y diferencia en VALIDACIÓN con su intervalo de confianza (tablas `10` y `11`); si la complejidad de las redes se justifica en estos datos.

### Referencias

- Bergstra, J. y Bengio, Y. (2012). Random Search for Hyper-Parameter Optimization. *Journal of Machine Learning Research*, 13, 281–305.
- Ke, G. et al. (2017). LightGBM: A Highly Efficient Gradient Boosting Decision Tree. *Advances in Neural Information Processing Systems*, 30.
- Grinsztajn, L., Oyallon, E. y Varoquaux, G. (2022). Why do tree-based models still outperform deep learning on typical tabular data? *NeurIPS, Datasets and Benchmarks Track*.